In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

client = genai.Client()
model = "gemini-3.1-flash-lite"

In [8]:
# Helper functions
def add_user_message(messages, message):
    if isinstance(message, list):
        parts = message
    else:
        parts = [{"text": message}]

    messages.append({
        "role": "user",
        "parts": parts,
    })


def add_assistant_message(messages, message):
    if isinstance(message, str):
        messages.append({
            "role": "model",
            "parts": [{"text": message}],
        })
    else:
        # `message` is already a types.Content (built from the streamed response)
        messages.append(message)


def chat_stream(messages, system=None, temperature=1.0, stop_sequences=None, tools=None):
    params = {
        "model": model,
        "contents": messages,
    }

    config_kwargs = {"temperature": temperature}
    if system:
        config_kwargs["system_instruction"] = system
    if stop_sequences:
        config_kwargs["stop_sequences"] = stop_sequences
    if tools:
        config_kwargs["tools"] = tools

    params["config"] = types.GenerateContentConfig(**config_kwargs)

    return client.models.generate_content_stream(**params)


def text_from_message(message):
    return "\n".join([part.text for part in message.parts if part.text is not None])

In [9]:
# Tool definition
save_article_schema = types.FunctionDeclaration(
    name="save_article",
    description="Saves a scholarly journal article",
    parameters={
        "type": "object",
        "properties": {
            "abstract": {
                "type": "string",
                "description": "Abstract of the article. One short sentence max",
            },
            "meta": {
                "type": "object",
                "properties": {
                    "word_count": {
                        "type": "integer",
                        "description": "Word count",
                    },
                    "review": {
                        "type": "string",
                        "description": "Eight sentence review of the paper",
                    },
                },
                "required": ["word_count", "review"],
            },
        },
        "required": ["abstract", "meta"],
    },
)

save_short_article_schema = types.FunctionDeclaration(
    name="save_article",
    description="Saves a scholarly journal article",
    parameters={
        "type": "object",
        "properties": {
            "abstract": {
                "type": "string",
                "description": "Abstract of the article. One short sentence max",
            },
            "meta": {
                "type": "object",
                "properties": {
                    "word_count": {
                        "type": "integer",
                        "description": "Word count",
                    },
                    "review": {
                        "type": "string",
                        "description": "Review of paper. One short sentence max",
                    },
                },
                "required": ["word_count", "review"],
            },
        },
        "required": ["abstract", "meta"],
    },
)


def save_article(**kwargs):
    return "Article saved!"

In [10]:
# Tool running
import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


def run_tools(message):
    tool_requests = [part for part in message.parts if part.function_call is not None]
    tool_result_parts = []

    for tool_request in tool_requests:
        function_call = tool_request.function_call
        try:
            tool_output = run_tool(function_call.name, dict(function_call.args))
            tool_result_parts.append({
                "function_response": {
                    "name": function_call.name,
                    "response": {"result": tool_output},
                }
            })
        except Exception as e:
            tool_result_parts.append({
                "function_response": {
                    "name": function_call.name,
                    "response": {"error": str(e)},
                }
            })

    return tool_result_parts

In [11]:
# Run conversation
def run_conversation(messages, tools=None):
    while True:
        stream = chat_stream(messages, tools=tools)

        text_buffer = ""
        function_call_parts = []

        for chunk in stream:
            content = chunk.candidates[0].content
            for part in content.parts:
                if part.text:
                    print(part.text, end="")
                    text_buffer += part.text

                if part.function_call:
                    print(f'>>> Tool Call: "{part.function_call.name}"')
                    print(json.dumps(dict(part.function_call.args), indent=2))
                    function_call_parts.append(part)

        print("")

        response_parts = function_call_parts
        if text_buffer:
            response_parts = [types.Part(text=text_buffer)] + function_call_parts

        response_content = types.Content(role="model", parts=response_parts)

        add_assistant_message(messages, response_content)

        if not function_call_parts:
            break

        tool_results = run_tools(response_content)
        add_user_message(messages, tool_results)

    return messages

In [12]:
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[types.Tool(function_declarations=[save_article_schema])],
)

>>> Tool Call: "save_article"
{
  "meta": {
    "review": "The authors present an intriguing approach to distributed machine learning under constrained bandwidth. They argue that stitching model layers across nodes reduces latency significantly compared to traditional federated learning. While the mathematical proofs provided are robust, the experimental validation lacks real-world edge device testing. The simulation environment used seems slightly too idealistic for noisy network conditions. However, the theoretical contribution to mitigating quantum-based intercept threats is commendable. This research serves as a solid foundation for future cryptographic neural network applications. Further studies should focus on the energy consumption trade-offs inherent in their stitching mechanism. Overall, it is a thought-provoking piece that bridges the gap between cybersecurity and edge efficiency.",
    "word_count": 1850
  },
  "abstract": "This paper introduces a novel algorithm, Quantum-R

[{'role': 'user',
  'parts': [{'text': 'Create and save a fake computer science article'}]},
 Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'abstract': 'This paper introduces a novel algorithm, Quantum-Resilient Neural Stitching, designed to optimize data transmission in decentralized edge computing architectures.',
           'meta': {
             'review': 'The authors present an intriguing approach to distributed machine learning under constrained bandwidth. They argue that stitching model layers across nodes reduces latency significantly compared to traditional federated learning. While the mathematical proofs provided are robust, the experimental validation lacks real-world edge device testing. The simulation environment used seems slightly too idealistic for noisy network conditions. However, the theoretical contribution to mitigating quantum-based intercept threats is commendable. This research serves as a solid foundation for futu